In [ ]:

# -*- coding: utf-8 -*-
"""
HoloTSH 逆袭大考引擎 (The David vs. Goliath Experiment) - 耐心 Debug 版
================================================================================
- 绝对禁止 Timeout：尊重 API 的自然延迟（一分钟一条也耐心等）。
- 还原昨天成功的重试机制（Exponential Backoff）。
- 逐条、顺次执行：1个Case -> Gemini -> GLM -> 算分 -> 打印明细 -> 下一个。
- Numpy 序列化崩盘与张量维度的修复已包含。
"""

import os
import json
import re
import time
import random
import requests
import numpy as np
import pandas as pd

from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --------------------- 绝密配置区 ---------------------
GEMINI_API_KEY = "YOUR_API_KEY_HERE"
ZHIPU_API_KEY = "YOUR_API_KEY_HERE"

TOP_K_LIST = [1, 3, 5]
ALPHA = 0.6
LLM_CANDIDATE_SIZE = 15
SLEEP_BETWEEN = 1.0
EXAM_SIZE = 100           # 提取 100 题先做测试

DRIVE_DIR = './data/'
GOLDEN_PATH = next((os.path.join(DRIVE_DIR, f) for f in ['SFT_Golden_Distilled_V3.3.json', 'SFT_Golden_Distilled.json'] if os.path.exists(os.path.join(DRIVE_DIR, f))), None)

# --------------------- 图谱与基础件 ---------------------
def build_prior_manifold():
    print("💎 [Step 1] 正在加载 SymMap 并构建先验拉普拉斯流形...")
    for file, url in [("SMTS.xlsx", "http://www.symmap.org/static/download/V2.0/SymMap%20v2.0%2C%20SMTS%20file.xlsx"),
                      ("SMSY.xlsx", "http://www.symmap.org/static/download/V2.0/SymMap%20v2.0%2C%20SMSY%20file.xlsx")]:
        if not os.path.exists(os.path.join(DRIVE_DIR, file)):
            open(os.path.join(DRIVE_DIR, file), 'wb').write(requests.get(url).content)

# Core HoloAuditor topological manifold embedding code is redacted pending peer review at Nature Machine Intelligence.

    return scores_detail

# --------------------- API 通讯 (还原昨天逻辑，不加 Timeout) ---------------------
def call_gemini(symptoms_str, top_k):
    # 还原昨天完全一致的 Prompt
    prompt = f"你是一个资深中医研究专家助手，在协助我测试中医。请根据以下患者的综合症状：[{symptoms_str}]，推荐最相关的 {top_k} 个中医证型（如气虚、血瘀、湿热等）。只输出证型名称，用逗号分隔，严禁解释。"
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent?key={GEMINI_API_KEY}"
    data = {"contents": [{"parts": [{"text": prompt}]}]}

    for attempt in range(3):
        try:
            # 绝对不加 timeout，耐心等待
            response = requests.post(url, headers={'Content-Type': 'application/json'}, json=data)
            if response.status_code == 200:
                res_json = response.json()
                if 'candidates' in res_json and res_json['candidates']:
                    candidate = res_json['candidates'][0]
                    if 'content' in candidate and 'parts' in candidate['content']:
                        text = candidate['content']['parts'][0]['text']
                        # 清洗数字和标点
                        text = re.sub(r'[\d\.\*]+', '', text)
                        return [n.strip() for n in re.split(r'[,，、\s\n]+', text) if len(n.strip()) >= 2][:top_k]
                return []
            elif response.status_code == 429:
                wait = 2 ** attempt
                print(f"      [Gemini 429 限流] 等待 {wait} 秒...")
                time.sleep(wait)
            else:
                print(f"      [Gemini 错误] HTTP {response.status_code}: {response.text[:100]}")
                return []
        except Exception as e:
            print(f"      [Gemini 异常] {str(e)[:50]}")
            if attempt < 2: time.sleep(2 ** attempt)
            else: return []
    return []

def call_glm(symptoms_str, top_k):
    # 还原昨天完全一致的 Prompt
    prompt = f"你是一个资深中医研究专家助手，在协助我测试中医。请根据以下患者的综合症状：[{symptoms_str}]，推荐最相关的 {top_k} 个中医证型（如气虚、血瘀、湿热等）。只输出证型名称，用逗号分隔，严禁解释。"
    url = "https://open.bigmodel.cn/api/paas/v4/chat/completions"
    headers = {"Authorization": f"Bearer {ZHIPU_API_KEY}", "Content-Type": "application/json"}
    data = {"model": "glm-4-flash", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}

    for attempt in range(3):
        try:
            # 绝对不加 timeout，耐心等待
            response = requests.post(url, headers=headers, json=data)
            if response.status_code == 200:
                text = response.json()['choices'][0]['message']['content']
                text = re.sub(r'[\d\.\*]+', '', text)
                return [n.strip() for n in re.split(r'[,，、\s\n]+', text) if len(n.strip()) >= 2][:top_k]
            elif response.status_code == 429:
                wait = 2 ** attempt
                print(f"      [GLM 429 限流] 等待 {wait} 秒...")
                time.sleep(wait)
            else:
                print(f"      [GLM 错误] HTTP {response.status_code}: {response.text[:100]}")
                return []
        except Exception as e:
            print(f"      [GLM 异常] {str(e)[:50]}")
            if attempt < 2: time.sleep(2 ** attempt)
            else: return []
    return []

# --------------------- 动态打印台 ---------------------
def print_radar(results, k=1):
    gl, g6, gl_llm, gl_6 = 0, 0, 0, 0
    for r in results:
        tgt = r['真实证型']
        if tgt in r['Gemini_LLM'][:k]: gl += 1
        if tgt in r['Gemini_V6.5'][:k]: g6 += 1
        if tgt in r['GLM_LLM'][:k]: gl_llm += 1
        if tgt in r['GLM_V6.5'][:k]: gl_6 += 1
    t = len(results)
    print(f"   [阶段统计 Hit@{k}] Gemini 裸跑: {gl/t*100:.1f}% -> 护栏: {g6/t*100:.1f}% | GLM 裸跑: {gl_llm/t*100:.1f}% -> 护栏: {gl_6/t*100:.1f}%")

# --------------------- 主程序 ---------------------
def main():
    print("="*80)
    print("🚀 HoloTSH 逆袭大考 (The David vs. Goliath) - 耐心 Debug 版")
    print("="*80)

    if not GOLDEN_PATH:
        print("❌ 错误：未找到 黄金数据集，请确认数据已生成！")
        return

    with open(GOLDEN_PATH, 'r', encoding='utf-8') as f:
        golden_raw = json.load(f)

    random.seed(42)
    exam_cases = random.sample(golden_raw, min(EXAM_SIZE, len(golden_raw)))
    print(f"📄 成功提取黄金考卷: {len(exam_cases)} 题")

    sym_to_syn, n2i, embeddings = build_prior_manifold()

    results = []
    checkpoint_path = os.path.join(DRIVE_DIR, 'Ablation_Exam_Results_Patient.json')

    print("\n⚔️ 考试开始：严格顺序执行 (1题 -> Gemini -> GLM -> 算分 -> 下1题)\n")
    for i, item in enumerate(exam_cases):
        symptoms = item.get('Extracted_Symptoms', [])
        true_syndrome = item.get('Topology_Target') or item.get('Target_Syndrome_Anchored') or item.get('Target_Syndrome')
        if not symptoms or not true_syndrome: continue
        sym_str = "、".join(symptoms)

        print(f"👉 [第 {i+1}/{len(exam_cases)} 题] 真实靶点: 【{true_syndrome}】 | 症状: {sym_str[:30]}...")

        # ---------------- 1. 呼叫 Gemini ----------------
        print("   🤖 正在呼叫 Gemini...", end=" ", flush=True)
        t0 = time.time()
        gemini_cands = call_gemini(sym_str, LLM_CANDIDATE_SIZE)
        print(f"耗时 {time.time()-t0:.1f}s -> 返回: {gemini_cands[:4] if gemini_cands else '空'}")
        time.sleep(SLEEP_BETWEEN) # 遵守规则，喘口气

        # ---------------- 2. 呼叫 GLM ----------------
        print("   🧑‍💻 正在呼叫 GLM-4...", end=" ", flush=True)
        t0 = time.time()
        glm_cands = call_glm(sym_str, LLM_CANDIDATE_SIZE)
        print(f"耗时 {time.time()-t0:.1f}s -> 返回: {glm_cands[:4] if glm_cands else '空'}")
        time.sleep(SLEEP_BETWEEN)

        # ---------------- 3. HoloTSH 算分与重排 ----------------
        scores_g = compute_scores(symptoms, gemini_cands, sym_to_syn, n2i, embeddings)
        gemini_v60 = sorted(gemini_cands, key=lambda x: scores_g[x]['cooc_raw'], reverse=True)
        gemini_v65 = sorted(gemini_cands, key=lambda x: scores_g[x]['final'], reverse=True)

        scores_z = compute_scores(symptoms, glm_cands, sym_to_syn, n2i, embeddings)
        glm_v60 = sorted(glm_cands, key=lambda x: scores_z[x]['cooc_raw'], reverse=True)
        glm_v65 = sorted(glm_cands, key=lambda x: scores_z[x]['final'], reverse=True)

        # ---------------- 4. 打印本题得分 Debug ----------------
        if gemini_cands:
            top1_g_llm = gemini_cands[0]
            top1_g_v65 = gemini_v65[0]
            print(f"      [诊断对比] Gemini 原生首选: {top1_g_llm} | 护栏重排首选: {top1_g_v65} (得分 {scores_g[top1_g_v65]['final']:.3f})")

        # 保存结果
        results.append({
            'UID': i+1, '症状列表': symptoms, '真实证型': true_syndrome,
            'Gemini_LLM': gemini_cands, 'Gemini_V6.0': gemini_v60, 'Gemini_V6.5': gemini_v65,
            'GLM_LLM': glm_cands, 'GLM_V6.0': glm_v60, 'GLM_V6.5': glm_v65
        })

        # ---------------- 5. 阶段性存盘与播报 ----------------
        if (i+1) % 10 == 0:
            print("-" * 60)
            print_radar(results, k=1)
            print("-" * 60)
            with open(checkpoint_path, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

    # 最终宣判
    if not results:
        print("\n⚠️ 考卷全部作废，请检查 API Key 或网络配置！")
        return

    print("\n" + "="*80)
    print("🏆 终极消融大考成绩单 (Final Leaderboard)")
    print("="*80)
    for k in TOP_K_LIST:
        print(f"\n🎯 Hit@{k} 准确率:")
        g_l = sum(1 for r in results if r['真实证型'] in r['Gemini_LLM'][:k]) / len(results)
        g_6 = sum(1 for r in results if r['真实证型'] in r['Gemini_V6.5'][:k]) / len(results)
        z_l = sum(1 for r in results if r['真实证型'] in r['GLM_LLM'][:k]) / len(results)
        z_6 = sum(1 for r in results if r['真实证型'] in r['GLM_V6.5'][:k]) / len(results)

        print(f"  🤖 万亿算力组 (Gemini) : 裸奔 {g_l*100:.1f}% -> 加装护栏 {g_6*100:.1f}% (+{(g_6-g_l)*100:.1f}%)")
        print(f"  🧑‍💻 开源百亿组 (GLM-4)  : 裸奔 {z_l*100:.1f}% -> 加装护栏 {z_6*100:.1f}% (+{(z_6-z_l)*100:.1f}%)")

if __name__ == "__main__":
    main()

Mounted at /content/drive
🚀 HoloTSH 逆袭大考 (The David vs. Goliath) - 耐心 Debug 版
📄 成功提取黄金考卷: 100 题
💎 [Step 1] 正在加载 SymMap 并构建先验拉普拉斯流形...

⚔️ 考试开始：严格顺序执行 (1题 -> Gemini -> GLM -> 算分 -> 下1题)

👉 [第 1/100 题] 真实靶点: 【阳虚】 | 症状: 头晕、发热、乏力、心慌、水肿、纳差...
   🤖 正在呼叫 Gemini... 耗时 46.3s -> 返回: ['脾气虚', '肾阳虚', '心血虚', '湿热内蕴']
   🧑‍💻 正在呼叫 GLM-4... 耗时 3.3s -> 返回: ['气虚', '血瘀', '湿热', '阴虚']
      [诊断对比] Gemini 原生首选: 脾气虚 | 护栏重排首选: 脾肾阳虚 (得分 0.725)
👉 [第 2/100 题] 真实靶点: 【热证】 | 症状: 脉弦、舌红、腹胀、结节...
   🤖 正在呼叫 Gemini... 耗时 20.5s -> 返回: ['肝气郁结', '气滞血瘀', '痰气互结', '肝郁化火']
   🧑‍💻 正在呼叫 GLM-4... 耗时 3.5s -> 返回: ['脉弦', '舌红', '腹胀', '结节']
      [诊断对比] Gemini 原生首选: 肝气郁结 | 护栏重排首选: 血瘀 (得分 0.028)
👉 [第 3/100 题] 真实靶点: 【气虚】 | 症状: 倦怠、汗出、面色晄白、倦怠乏力、心悸、胸闷、咳嗽、乏力、痛、...
   🤖 正在呼叫 Gemini... 耗时 27.7s -> 返回: ['气虚', '肺气虚', '心气虚', '阳虚']
   🧑‍💻 正在呼叫 GLM-4... 耗时 2.1s -> 返回: ['气虚', '血瘀', '湿热', '阳虚']
      [诊断对比] Gemini 原生首选: 气虚 | 护栏重排首选: 气虚 (得分 0.732)
👉 [第 4/100 题] 真实靶点: 【热证】 | 症状: 脉弦、反酸、舌红、嗳气、苔黄...
   🤖 正在呼叫 Gemini... 耗时 26.2s -> 返回: ['肝胃不和', '肝气犯胃', '肝火犯

In [ ]:
# -*- coding: utf-8 -*-
"""
全量扫描 SFT 和 ShenNong 数据集：V6.0 审计（基于共现）与 V6.5 审计（基于拓扑）
- 对每个原始病例，提取症状（否定词防御）和方剂（匹配经典方剂或通用正则）
- V6.0：基于全局证型-方剂共现判断方剂是否常用
- V6.5：基于图嵌入计算拓扑得分，输出得分和推断证型
- 输出 CSV 文件，供后续分析
"""

import os
import re
import json
import requests
import gdown
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from datasets import load_dataset


# ================== 配置 ==================
OUTPUT_DIR = "./data/"
TOP_K_COMMON = 5           # 常用方剂列表的 TOP K
ALPHA = 0.3                # V6.5 融合参数
SFT_MAX = None             # 限制 SFT 处理数量（None 表示全部）
SHENNONG_MAX = 50000       # ShenNong 处理数量（与之前一致）

# 经典方剂列表（可扩展）
GOLDEN_DICT = {
    '太阳中风': '桂枝汤', '太阳伤寒': '麻黄汤', '少阳病': '小柴胡汤', '阳明腑实': '大承气汤',
    '阳明经': '白虎汤', '太阴病': '理中汤', '少阴寒化': '四逆汤', '厥阴病': '乌梅丸',
    '气血两虚': '八珍汤', '肝郁脾虚': '逍遥散', '肝胆湿热': '龙胆泻肝汤', '心火亢盛': '导赤散',
    '肺热壅盛': '麻杏石甘汤', '脾胃虚寒': '附子理中丸', '肾阴虚': '六味地黄丸', '肾阳虚': '金匮肾气丸',
    '风寒表实': '荆防败毒散', '痰热蕴肺': '清气化痰丸', '瘀血阻络': '血府逐瘀汤', '肝阳上亢': '天麻钩藤饮'
}
CLASSIC_FORMULAS = set(GOLDEN_DICT.values())

# 证型拆解表（用于复合证型）
SYNDROME_DECOMPOSE = {
    "气虚血瘀证": ["气虚", "血瘀"],
    "湿热下注证": ["湿热"],
    "正虚瘀结证": ["正虚", "瘀结"],
    "热毒蕴结证": ["热毒"],
    "风痰瘀阻证": ["风痰", "瘀阻"],
    "风寒外袭证": ["风寒"],
    "肾虚证": ["肾虚"],
    "气滞血瘀证": ["气滞", "血瘀"],
    "痰火扰心证": ["痰火"],
    "湿热蕴结证": ["湿热"],
    "风寒袭肺证": ["风寒"],
    "正虚毒瘀证": ["正虚", "毒瘀"],
}

# ================== 否定词防御症状提取 ==================
def extract_symptoms_safely(text, symptom_set):
    valid = []
    for sym in symptom_set:
        if sym in text:
            negation_pattern = r'([无不未非没]|否认)[^，。；：,.;:!?]*?' + re.escape(sym)
            if not re.search(negation_pattern, text):
                valid.append(sym)
    return valid

# ================== 方剂提取 ==================
def extract_formula(text):
    """优先匹配经典方剂，再通用正则"""
    for f in CLASSIC_FORMULAS:
        if f in text:
            return f
    pattern = r'[一二三四五六七八九十]?[\u4e00-\u9fa5]{2,6}(?:汤|丸|散|膏|丹|剂)'
    match = re.search(pattern, text)
    if match:
        return match.group()
    return None

# ================== 加载 SymMap 并构建嵌入 ==================
def load_symmap_and_embedding():
    print("\n加载 SymMap 并构建拓扑流形...")
    def download(url, local):
        if not os.path.exists(local):
            print(f"下载 {local} ...")
            r = requests.get(url)
            with open(local, 'wb') as f:
                f.write(r.content)
    download("http://www.symmap.org/static/download/V2.0/SymMap%20v2.0%2C%20SMTS%20file.xlsx", "SMTS.xlsx")
    download("http://www.symmap.org/static/download/V2.0/SymMap%20v2.0%2C%20SMSY%20file.xlsx", "SMSY.xlsx")

    df_sym = pd.read_excel("SMTS.xlsx", sheet_name='Sheet1').dropna(subset=['TCM_symptom_name', 'Symptom_property'])
    symptom_list = df_sym['TCM_symptom_name'].tolist()
    symptom_prop = {}

 # Core HoloAuditor topological manifold embedding code is redacted pending peer review at Nature Machine Intelligence.

    return symptom_list, syndrome_names, symptom_to_basic, node_to_idx, embeddings

def symptoms_to_vector(symptoms, node_to_idx, embeddings):
    valid = [node_to_idx[s] for s in symptoms if s in node_to_idx]
    if not valid:
        return None
    vecs = embeddings[valid]
    return np.mean(vecs, axis=0)

def infer_syndrome_topology(symptoms, node_to_idx, embeddings, syndrome_names):
    """基于拓扑相似度推断证型"""
    sym_vec = symptoms_to_vector(symptoms, node_to_idx, embeddings)
    if sym_vec is None:
        return None, 0.0
    sym_vec = sym_vec / np.linalg.norm(sym_vec)
    best_score = -1
    best_syn = None
    for syn in syndrome_names:
        if syn in node_to_idx:
            syn_vec = embeddings[node_to_idx[syn]]
            syn_vec = syn_vec / np.linalg.norm(syn_vec)
            score = np.dot(sym_vec, syn_vec)
            if score > best_score:
                best_score = score
                best_syn = syn
    return best_syn, best_score

def infer_syndrome_cooccur(symptoms, symptom_to_basic, syndrome_names):
    """基于症状共现（基本证素投票）推断证型"""
    basic_votes = defaultdict(int)
    for sym in symptoms:
        for basic in symptom_to_basic.get(sym, []):
            basic_votes[basic] += 1
    if not basic_votes:
        return None
    # 找到投票最多的基本证素，再映射到包含该基本证素的原始证型
    top_basic = max(basic_votes.items(), key=lambda x: x[1])[0]
    # 简单起见，返回包含该基本证素的第一个证型（可能有多个，这里取第一个）
    for syn in syndrome_names:
        if syn in SYNDROME_DECOMPOSE:
            if top_basic in SYNDROME_DECOMPOSE[syn]:
                return syn
        elif syn == top_basic:
            return syn
    return None

# ================== 加载数据 ==================
def load_sft_data(symptom_list):
    print("\n加载 SFT 数据...")
    FILE_ID = "SFTdatasets"
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    output = "SFT_data.json"
    if not os.path.exists(output):
        gdown.download(url, output, quiet=False)
    with open(output, 'r') as f:
        data = json.load(f)
    if SFT_MAX:
        data = data[:SFT_MAX]
    symptom_set = set(symptom_list)
    cases = []
    for item in data:
        text = item.get('input', '') + ' ' + item.get('output', '')
        symptoms = extract_symptoms_safely(text, symptom_set)
        if not symptoms:
            continue
        formula = extract_formula(text)
        if not formula:
            continue
        cases.append((symptoms, formula, item))
    print(f"提取到 {len(cases)} 个 SFT 病例")
    return cases

def load_shennong_data(symptom_list, max_cases=50000):
    print("\n加载 ShenNong 数据...")
    ds = load_dataset("michaelwzhu/ShenNong_TCM_Dataset", split="train", streaming=True)
    symptom_set = set(symptom_list)
    cases = []
    for i, item in enumerate(ds):
        text = item.get('query', '') + ' ' + item.get('response', '')
        symptoms = extract_symptoms_safely(text, symptom_set)
        if not symptoms:
            continue
        formula = extract_formula(text)
        if not formula:
            continue
        cases.append((symptoms, formula, item))
        if max_cases and len(cases) >= max_cases:
            break
    print(f"提取到 {len(cases)} 个 ShenNong 病例")
    return cases

# ================== V6.0 审计 ==================
def audit_v6(cases, symptom_to_basic, syndrome_names):
    print("运行 V6.0 审计...")
    # 第一步：构建全局证型-方剂共现计数器
    global_counter = Counter()
    case_info = []  # (推断证型, 方剂)
    for symptoms, formula, _ in cases:
        syn = infer_syndrome_cooccur(symptoms, symptom_to_basic, syndrome_names)
        if syn is None:
            continue
        case_info.append((syn, formula))
        global_counter[(syn, formula)] += 1

    # 第二步：对每个病例，减去自身贡献后判断
    results = []
    for (symptoms, formula, item), (syn, _) in zip(cases, case_info):
        # 临时计数器
        temp_counter = global_counter.copy()
        if temp_counter[(syn, formula)] > 0:
            temp_counter[(syn, formula)] -= 1
            if temp_counter[(syn, formula)] == 0:
                del temp_counter[(syn, formula)]

        # 获取该证型下的所有方剂频次
        syn_pres = {}
        for (s, p), count in temp_counter.items():
            if s == syn:
                syn_pres[p] = count
        sorted_pres = sorted(syn_pres.items(), key=lambda x: x[1], reverse=True)
        common_pres = [p for p, _ in sorted_pres[:TOP_K_COMMON]]
        is_suspicious = formula not in syn_pres

        results.append({
            '症状': ';'.join(symptoms),
            '方剂': formula,
            '推断证型': syn,
            '常用方剂': ';'.join(common_pres),
            '是否可疑': '是' if is_suspicious else '否'
        })
    return pd.DataFrame(results)

# ================== V6.5 审计 ==================
def audit_v65(cases, node_to_idx, embeddings, syndrome_names):
    print("运行 V6.5 审计...")
    results = []
    for symptoms, formula, _ in cases:
        syn, score = infer_syndrome_topology(symptoms, node_to_idx, embeddings, syndrome_names)
        if syn is None:
            continue
        # 这里可以根据 score 设定是否可靠，但全扫描不设阈值，都输出
        results.append({
            '症状': ';'.join(symptoms),
            '方剂': formula,
            '推断证型': syn,
            '拓扑得分': round(score, 4),
            '可靠': score >= 0.6  # 标记是否达到黄金标准（可选）
        })
    return pd.DataFrame(results)

# ================== 主程序 ==================
def main():
    print("="*80)
    print("全量扫描 SFT 和 ShenNong 数据集：V6.0 vs V6.5")
    print("="*80)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. 加载 SymMap 和嵌入
    symptom_list, syndrome_names, symptom_to_basic, node_to_idx, embeddings = load_symmap_and_embedding()

    # 2. 加载数据
    sft_cases = load_sft_data(symptom_list)
    sn_cases = load_shennong_data(symptom_list, max_cases=SHENNONG_MAX)

    # 3. 运行 V6.0 审计
    print("\n" + "="*40)
    print("SFT V6.0 审计")
    sft_v6_df = audit_v6(sft_cases, symptom_to_basic, syndrome_names)
    sft_v6_df.to_csv(os.path.join(OUTPUT_DIR, 'sft_audit_v6.csv'), index=False, encoding='utf-8-sig')
    print(f"SFT V6.0 审计完成，共 {len(sft_v6_df)} 条记录，保存至 sft_audit_v6.csv")

    print("\n" + "="*40)
    print("ShenNong V6.0 审计")
    sn_v6_df = audit_v6(sn_cases, symptom_to_basic, syndrome_names)
    sn_v6_df.to_csv(os.path.join(OUTPUT_DIR, 'shennong_audit_v6.csv'), index=False, encoding='utf-8-sig')
    print(f"ShenNong V6.0 审计完成，共 {len(sn_v6_df)} 条记录，保存至 shennong_audit_v6.csv")

    # 4. 运行 V6.5 审计
    print("\n" + "="*40)
    print("SFT V6.5 审计")
    sft_v65_df = audit_v65(sft_cases, node_to_idx, embeddings, syndrome_names)
    sft_v65_df.to_csv(os.path.join(OUTPUT_DIR, 'sft_audit_v65.csv'), index=False, encoding='utf-8-sig')
    print(f"SFT V6.5 审计完成，共 {len(sft_v65_df)} 条记录，保存至 sft_audit_v65.csv")

    print("\n" + "="*40)
    print("ShenNong V6.5 审计")
    sn_v65_df = audit_v65(sn_cases, node_to_idx, embeddings, syndrome_names)
    sn_v65_df.to_csv(os.path.join(OUTPUT_DIR, 'shennong_audit_v65.csv'), index=False, encoding='utf-8-sig')
    print(f"ShenNong V6.5 审计完成，共 {len(sn_v65_df)} 条记录，保存至 shennong_audit_v65.csv")

    print("\n✅ 全量扫描完成！")

if __name__ == "__main__":
    main()

全量扫描 SFT 和 ShenNong 数据集：V6.0 vs V6.5

加载 SymMap 并构建拓扑流形...
图节点数: 1851，嵌入维度: 64

加载 SFT 数据...
提取到 4597 个 SFT 病例

加载 ShenNong 数据...
提取到 50000 个 ShenNong 病例

SFT V6.0 审计
运行 V6.0 审计...
SFT V6.0 审计完成，共 4194 条记录，保存至 sft_audit_v6.csv

ShenNong V6.0 审计
运行 V6.0 审计...
ShenNong V6.0 审计完成，共 42115 条记录，保存至 shennong_audit_v6.csv

SFT V6.5 审计
运行 V6.5 审计...
SFT V6.5 审计完成，共 4482 条记录，保存至 sft_audit_v65.csv

ShenNong V6.5 审计
运行 V6.5 审计...
ShenNong V6.5 审计完成，共 48973 条记录，保存至 shennong_audit_v65.csv

✅ 全量扫描完成！


In [ ]:
# -*- coding: utf-8 -*-
"""
全量扫描结果分析：V6.0 vs V6.5
- 输入：四个CSV文件（sft_audit_v6.csv, sft_audit_v65.csv, shennong_audit_v6.csv, shennong_audit_v65.csv）
- 输出：统计报告、分类后的合并数据、差异案例详情、按得分区间抽样打印
"""

import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "./data/"

def load_and_merge(dataset_name):
    """加载并合并V6.0和V6.5的审计结果"""
    v6_path = os.path.join(OUTPUT_DIR, f'{dataset_name}_audit_v6.csv')
    v65_path = os.path.join(OUTPUT_DIR, f'{dataset_name}_audit_v65.csv')

    df_v6 = pd.read_csv(v6_path)
    df_v65 = pd.read_csv(v65_path)

    # 确保列名一致
    print(f"\n{dataset_name} V6.0 列名: {df_v6.columns.tolist()}")
    print(f"{dataset_name} V6.5 列名: {df_v65.columns.tolist()}")

    # 按症状和方剂合并
    merged = pd.merge(df_v65, df_v6, on=['症状', '方剂'], how='inner', suffixes=('_v65', '_v6'))
    print(f"{dataset_name}: V6.0 {len(df_v6)}条, V6.5 {len(df_v65)}条, 合并后 {len(merged)}条")
    return merged

def classify(row):
    """根据V6.0的“是否可疑”和V6.5的“可靠”进行分类"""
    v6_pass = str(row['是否可疑']).strip() == '否'
    # V6.5的“可靠”可能是布尔值或字符串'True'/'False'
    v65_pass_val = row['可靠']
    if isinstance(v65_pass_val, bool):
        v65_pass = v65_pass_val
    else:
        v65_pass = str(v65_pass_val).strip().lower() == 'true'

    if v6_pass and v65_pass:
        return '两者都通过'
    elif v6_pass and not v65_pass:
        return 'V6.0通过_V6.5不通过'
    elif not v6_pass and v65_pass:
        return 'V6.5通过_V6.0不通过'
    else:
        return '两者都不通过'

def print_samples_by_score(df, dataset_name, sample_count=5):
    """按得分区间抽样打印"""
    if df.empty:
        return

    # 确保拓扑得分列存在
    if '拓扑得分' not in df.columns:
        print(f"{dataset_name} 缺少拓扑得分列，跳过抽样打印。")
        return

    print(f"\n===== {dataset_name} 抽样验证 (各区间 {sample_count} 条) =====")

    # 定义区间
    bins = [0, 0.4, 0.6, 1.0]
    labels = ['低分 (<0.4)', '中分 (0.4-0.6)', '高分 (≥0.6)']
    df['得分区间'] = pd.cut(df['拓扑得分'], bins=bins, labels=labels, right=False)

    for interval in labels:
        subset = df[df['得分区间'] == interval]
        if subset.empty:
            print(f"\n【{interval}】无样本")
            continue
        samples = subset.sample(min(sample_count, len(subset)))
        print(f"\n【{interval}】共 {len(subset)} 条，抽样 {len(samples)} 条：")
        for idx, row in samples.iterrows():
            print(f"\n  病例 {idx}")
            print(f"  症状: {row['症状'][:80]}..." if len(row['症状']) > 80 else f"  症状: {row['症状']}")
            print(f"  方剂: {row['方剂']}")
            print(f"  V6.0推断证型: {row['推断证型_v6']}, 是否可疑: {row['是否可疑']}")
            print(f"  V6.5推断证型: {row['推断证型_v65']}, 拓扑得分: {row['拓扑得分']:.4f}, 可靠: {row['可靠']}")
            print(f"  差异类别: {row['差异类别']}")

def main():
    print("="*80)
    print("全量扫描结果分析：V6.0 vs V6.5")
    print("="*80)

    # 合并两个数据集
    sft_merged = load_and_merge('sft')
    sn_merged = load_and_merge('shennong')

    # 添加分类列
    sft_merged['差异类别'] = sft_merged.apply(classify, axis=1)
    sn_merged['差异类别'] = sn_merged.apply(classify, axis=1)

    # 统计
    print("\n" + "="*60)
    print("统计结果")
    print("="*60)
    print("\n【SFT 数据集】")
    sft_counts = sft_merged['差异类别'].value_counts()
    print(sft_counts)
    print("\n【ShenNong 数据集】")
    sn_counts = sn_merged['差异类别'].value_counts()
    print(sn_counts)

    # 保存合并后的数据
    sft_merged.to_csv(os.path.join(OUTPUT_DIR, 'sft_comparison_full.csv'), index=False, encoding='utf-8-sig')
    sn_merged.to_csv(os.path.join(OUTPUT_DIR, 'shennong_comparison_full.csv'), index=False, encoding='utf-8-sig')
    print("\n完整对比结果已保存至 sft_comparison_full.csv 和 shennong_comparison_full.csv")

    # 抽样打印（按得分区间）
    print_samples_by_score(sft_merged, 'SFT', sample_count=5)
    print_samples_by_score(sn_merged, 'ShenNong', sample_count=5)

    # 输出差异案例（V6.0通过_V6.5不通过 和 V6.5通过_V6.0不通过）
    for name, df in [('SFT', sft_merged), ('ShenNong', sn_merged)]:
        for diff_type in ['V6.0通过_V6.5不通过', 'V6.5通过_V6.0不通过']:
            cases = df[df['差异类别'] == diff_type]
            if not cases.empty:
                print(f"\n===== {name} 数据集：{diff_type} 案例（共 {len(cases)} 个）=====")
                # 显示前10个
                for idx, row in cases.head(10).iterrows():
                    print(f"\n病例 {idx}")
                    print(f"症状: {row['症状'][:100]}..." if len(row['症状']) > 100 else row['症状'])
                    print(f"方剂: {row['方剂']}")
                    print(f"V6.0 推断证型: {row['推断证型_v6']}, 是否可疑: {row['是否可疑']}")
                    print(f"V6.5 推断证型: {row['推断证型_v65']}, 拓扑得分: {row['拓扑得分']:.4f}, 可靠: {row['可靠']}")
                    print("-" * 40)
            else:
                print(f"\n{name} 没有 {diff_type} 的案例。")

if __name__ == "__main__":
    main()

全量扫描结果分析：V6.0 vs V6.5

sft V6.0 列名: ['症状', '方剂', '推断证型', '常用方剂', '是否可疑']
sft V6.5 列名: ['症状', '方剂', '推断证型', '拓扑得分', '可靠']
sft: V6.0 4194条, V6.5 4482条, 合并后 39635条

shennong V6.0 列名: ['症状', '方剂', '推断证型', '常用方剂', '是否可疑']
shennong V6.5 列名: ['症状', '方剂', '推断证型', '拓扑得分', '可靠']
shennong: V6.0 42115条, V6.5 48973条, 合并后 209684条

统计结果

【SFT 数据集】
差异类别
两者都不通过            20971
V6.0通过_V6.5不通过    12245
V6.5通过_V6.0不通过     4354
两者都通过              2065
Name: count, dtype: int64

【ShenNong 数据集】
差异类别
两者都通过             111013
V6.0通过_V6.5不通过     86717
两者都不通过              6121
V6.5通过_V6.0不通过      5833
Name: count, dtype: int64

完整对比结果已保存至 sft_comparison_full.csv 和 shennong_comparison_full.csv

===== SFT 抽样验证 (各区间 5 条) =====

【低分 (<0.4)】共 1250 条，抽样 5 条：

  病例 5959
  症状: 腹泻;结节;乏力;便血
  方剂: 具体剂
  V6.0推断证型: 脾虚, 是否可疑: 否
  V6.5推断证型: 肝肾亏虚, 拓扑得分: 0.3955, 可靠: False
  差异类别: V6.0通过_V6.5不通过

  病例 32189
  症状: 大便干燥;便干;大便干;便血
  方剂: 自用痔疮栓膏
  V6.0推断证型: 湿热中阻, 是否可疑: 否
  V6.5推断证型: 肝肾亏虚, 拓扑得分: 0.3589, 可靠: False
  差异类别: V6.0通过_V6.5不通

In [ ]:
# -*- coding: utf-8 -*-
"""
合并分析 V6.0 与 V6.5 审计结果（修正版）
- 使用正确的列名：V6.0 用 '是否可疑'，V6.5 用 '可靠'
"""

import os
import pandas as pd

OUTPUT_DIR = "./data/"

def safe_read_csv(path):
    if not os.path.exists(path):
        print(f"文件不存在: {path}")
        return None
    try:
        df = pd.read_csv(path)
        if df.empty:
            print(f"文件为空: {path}")
            return None
        return df
    except Exception as e:
        print(f"读取文件失败 {path}: {e}")
        return None

def analyze_dataset(name, df_v6, df_v65):
    if df_v6 is None or df_v65 is None:
        print(f"跳过 {name} 分析，因为缺少数据。")
        return

    # 按 (症状, 方剂) 合并
    merged = pd.merge(df_v65, df_v6, on=['症状', '方剂'], how='inner', suffixes=('_v65', '_v6'))
    print(f"{name} 合并后记录数: {len(merged)}")

    if merged.empty:
        print(f"{name} 无合并记录。")
        return

    # 定义分类函数
    def classify(row):
        # V6.0 通过条件：'是否可疑' 为 '否'
        v6_pass = str(row['是否可疑']).strip() == '否'
        # V6.5 通过条件：'可靠' 为 True（可能是布尔或字符串 'True'）
        v65_pass = str(row['可靠']).strip().lower() == 'true'

        if v6_pass and v65_pass:
            return '两者都通过'
        elif v6_pass and not v65_pass:
            return 'V6.0通过_V6.5不通过'
        elif not v6_pass and v65_pass:
            return 'V6.5通过_V6.0不通过'
        else:
            return '两者都不通过'

    merged['差异类别'] = merged.apply(classify, axis=1)

    # 统计
    counts = merged['差异类别'].value_counts()
    print(f"\n【{name} 差异统计】")
    print(counts)

    # 输出差异案例详情（只显示前10个）
    for diff_type in ['V6.0通过_V6.5不通过', 'V6.5通过_V6.0不通过']:
        cases = merged[merged['差异类别'] == diff_type]
        if not cases.empty:
            print(f"\n===== {name} 数据集：{diff_type} 案例（共 {len(cases)} 个，显示前10个）=====")
            for idx, row in cases.head(10).iterrows():
                print(f"\n病例 {idx}")
                symptoms = row['症状']
                print(f"症状: {symptoms[:100]}..." if len(symptoms) > 100 else symptoms)
                print(f"方剂: {row['方剂']}")
                print(f"V6.0 推断证型: {row['推断证型_v6']}, 是否可疑: {row['是否可疑']}")
                print(f"V6.5 推断证型: {row['推断证型_v65']}, 拓扑得分: {row['拓扑得分']:.4f}, 可靠: {row['可靠']}")
                print("-" * 40)
        else:
            print(f"\n{name} 没有 {diff_type} 的案例。")

    # 保存合并后的完整数据
    out_path = os.path.join(OUTPUT_DIR, f'{name}_comparison_full.csv')
    merged.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n{name} 完整对比结果已保存至 {out_path}")

def main():
    print("="*60)
    print("合并分析 V6.0 与 V6.5 审计结果（修正版）")
    print("="*60)

    # 读取文件
    sft_v6 = safe_read_csv(os.path.join(OUTPUT_DIR, 'sft_audit_v6.csv'))
    sft_v65 = safe_read_csv(os.path.join(OUTPUT_DIR, 'sft_audit_v65.csv'))
    sn_v6 = safe_read_csv(os.path.join(OUTPUT_DIR, 'shennong_audit_v6.csv'))
    sn_v65 = safe_read_csv(os.path.join(OUTPUT_DIR, 'shennong_audit_v65.csv'))

    # 分析 SFT
    analyze_dataset('SFT', sft_v6, sft_v65)

    # 分析 ShenNong
    analyze_dataset('ShenNong', sn_v6, sn_v65)

if __name__ == "__main__":
    main()

合并分析 V6.0 与 V6.5 审计结果（修正版）
SFT 合并后记录数: 39635

【SFT 差异统计】
差异类别
两者都不通过            20971
V6.0通过_V6.5不通过    12245
V6.5通过_V6.0不通过     4354
两者都通过              2065
Name: count, dtype: int64

===== SFT 数据集：V6.0通过_V6.5不通过 案例（共 12245 个，显示前10个）=====

病例 13
肌肉酸痛;白细胞减少;鼻塞;痛;痒;疼痛;感冒;流涕
方剂: 双藤宣痹丸
V6.0 推断证型: 脾虚, 是否可疑: 否
V6.5 推断证型: 气郁, 拓扑得分: 0.2797, 可靠: False
----------------------------------------

病例 15
痛;湿热下注;便血
方剂: 自用痔疮栓膏
V6.0 推断证型: 湿热中阻, 是否可疑: 否
V6.5 推断证型: 肝肾亏虚, 拓扑得分: 0.4386, 可靠: False
----------------------------------------

病例 20
腹泻;腹胀;结节;便干;大便干结;痛;便秘;大便干;腹痛;呕吐
方剂: 具体药物剂
V6.0 推断证型: 津亏, 是否可疑: 否
V6.5 推断证型: 积滞不消, 拓扑得分: 0.5155, 可靠: False
----------------------------------------

病例 32
自汗;汗出;头晕;乏力;倦怠乏力;面色晄白;倦怠
方剂: 含服速效救心丸
V6.0 推断证型: 气逆, 是否可疑: 否
V6.5 推断证型: 阴虚, 拓扑得分: 0.4286, 可靠: False
----------------------------------------

病例 40
腹泻;结节;便血
方剂: 具体剂
V6.0 推断证型: 肾阳虚, 是否可疑: 否
V6.5 推断证型: 肝肾亏虚, 拓扑得分: 0.4311, 可靠: False
----------------------------------------

病例 41
腹泻;结节;便血
方剂: 具体剂
V6.0 推断证型: 湿热中阻, 是否可疑: 

In [ ]:

# -*- coding: utf-8 -*-
"""
HoloTSH V7.1: 全自動繁簡歸一 + 糾偏大考引擎
================================================================================
- 核心：解決 Gemini 3.1 Pro 繁體輸出導致的 0 分匹配問題
- 修正：自動將所有輸出轉為簡體，對齊 SymMap 數據庫
- 目標：獲取 3.1 Pro 與 V3.3 Signature Approach 的真實對比數據
"""

# 1. 安裝繁簡轉換庫
!pip install opencc-python-reimplemented -q

import os, json, re, time, random, requests, sys
import numpy as np
import pandas as pd
# Core HoloAuditor topological manifold embedding code is redacted pending peer review at Nature Machine Intelligence.
from collections import defaultdict
from opencc import OpenCC
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --------------------- 配置區 ---------------------
GEMINI_API_KEY = "YOUR_API_KEY_HERE"
DRIVE_DIR = './data/'
JSON_CHECKPOINT = os.path.join(DRIVE_DIR, 'Ablation_Results_Gemini31_V71.json')
TXT_LOG_FILE = os.path.join(DRIVE_DIR, 'Gemini31_Audit_Log_V71.txt')

MODEL_NAME = "gemini-3.1-pro-preview"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent?key={GEMINI_API_KEY}"

ALPHA = 0.6
LLM_CANDIDATE_SIZE = 20
EXAM_SIZE = 100
t2s = OpenCC('t2s') # 繁體轉簡體對象

# --------------------- 1. 雙路日誌 ---------------------
class TeeLogger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", encoding="utf-8") # 每次重跑覆蓋，確保乾淨
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.terminal.flush(); self.log.flush()
    def flush(self): pass

sys.stdout = TeeLogger(TXT_LOG_FILE)

# --------------------- 2. 算法引擎 (修正版) ---------------------
def build_manifold():
    print(f"\n💎 [Step 1] 加載流形數據庫 (V7.1 繁簡兼容版)...", flush=True)

  # Core HoloAuditor topological manifold embedding code is redacted pending peer review at Nature Machine Intelligence.


    return symptom_to_syndromes, n2i, embeddings

def compute_scores(symptoms, candidates, sym_to_syn, n2i, embeddings):
    # 這裡 candidates 已經在外部被 call_gemini_31 轉為簡體了
    vecs, weights = [], []
    for s in symptoms:
        if s in n2i:
            vecs.append(embeddings[n2i[s]])
            weights.append(5.0 if any(k in s for k in ['舌', '脉', '苔']) else 1.0)
    sym_vec = np.asarray(np.average(vecs, axis=0, weights=weights)).flatten() if vecs else None

    syn_scores = defaultdict(float)
    for sym in symptoms:
        for syn in sym_to_syn.get(sym, []): syn_scores[syn] += 1.0

    scores_detail = {}
    for cand in candidates:
        cooc = float(syn_scores.get(cand, 0.0))
        topo = 0.0
        if sym_vec is not None and cand in n2i:
            syn_vec = np.asarray(embeddings[n2i[cand]]).flatten()
            n_s, n_t = np.linalg.norm(sym_vec), np.linalg.norm(syn_vec)
            if n_s > 1e-8 and n_t > 1e-8: topo = float(np.dot(sym_vec, syn_vec) / (n_s * n_t))
        scores_detail[cand] = {'cooc_raw': float(cooc), 'topo': float(topo), 'final': 0.0}

    max_cooc = max((s['cooc_raw'] for s in scores_detail.values()), default=1.0) or 1.0
    for cand, d in scores_detail.items():
        d['cooc_norm'] = float(d['cooc_raw'] / max_cooc)
        d['final'] = float(ALPHA * d['topo'] + (1 - ALPHA) * d['cooc_norm'])
    return scores_detail

# --------------------- 3. API & 洗脫 (強制簡體) ---------------------
def call_gemini_31(symptoms_str, top_k):
    prompt_text = (f"你是一個資深中醫。症狀：[{symptoms_str}]。請推薦最相關的 {top_k} 個中醫證型。"
                   f"要求：只輸出簡體中文標準名稱（如'气虚'、'肝火上炎'），禁止輸出英文，禁止加'证'字。"
                   f"請用逗號分隔。")
    prompt = {"contents": [{"parts": [{"text": prompt_text}]}], "generationConfig": {"temperature": 0.1}}
    try:
        resp = requests.post(API_URL, json=prompt, timeout=60)
        if resp.status_code == 200:
            text = resp.json()['candidates'][0]['content']['parts'][0]['text']
            # 強制繁轉簡
            text_simplified = t2s.convert(text)
            # 洗脫後綴與符號
            text_simplified = text_simplified.replace('证', '').replace('症', '')
            cands = [n.strip() for n in re.split(r'[,，、\s\n]+', re.sub(r'[\d\.\*]+', '', text_simplified)) if len(n.strip()) >= 2]
            cands = [c for c in cands if re.search(r'[\u4e00-\u9fa5]', c)]
            return cands[:top_k]
    except: pass
    return []

# --------------------- 4. 大考程序 ---------------------
def main():
    print("="*80)
    print(f"🚀 HoloTSH V7.1: {MODEL_NAME} David vs. Goliath Final Trial")
    print("="*80)

    results = []
    golden_path = next((os.path.join(DRIVE_DIR, f) for f in ['SFT_Golden_Distilled_V3.3.json', 'SFT_Golden_Distilled.json'] if os.path.exists(os.path.join(DRIVE_DIR, f))), None)
    with open(golden_path, 'r', encoding='utf-8') as f:
        golden_raw = json.load(f)

    random.seed(42)
    exam_cases = random.sample(golden_raw, min(EXAM_SIZE, len(golden_raw)))
    sym_to_syn, n2i, embeddings = build_manifold()

    for i, item in enumerate(exam_cases):
        symptoms = item.get('Extracted_Symptoms', [])
        true_syndrome = item.get('Topology_Target') or item.get('Target_Syndrome_Anchored') or item.get('Target_Syndrome')
        if not symptoms or not true_syndrome: continue
        # Target 也要簡體化匹配
        true_syndrome = t2s.convert(true_syndrome).replace('证', '').replace('症', '')

        print(f"\n👉 [處理 {i+1}/100] 靶點: 【{true_syndrome}】")

        g31_clean = call_gemini_31("、".join(symptoms), LLM_CANDIDATE_SIZE)
        print(f"    {MODEL_NAME} 返回 (簡體歸一化): {g31_clean}")

        scores = compute_scores(symptoms, g31_clean, sym_to_syn, n2i, embeddings)
        g31_holotsh = sorted(g31_clean, key=lambda x: scores[x]['final'], reverse=True)

        print(f"    [詳細評分 Debug]")
        for cand in g31_clean:
            sc = scores[cand]
            print(f"       {cand}: cooc_raw={sc['cooc_raw']:.2f}, topo={sc['topo']:.3f}, final={sc['final']:.3f}")

        results.append({
            'UID': i+1, '真實证型': true_syndrome,
            'Native_LLM': g31_clean, 'HoloTSH_V6.7': g31_holotsh
        })

        if (i+1) % 5 == 0:
            acc_raw = sum(1 for r in results if r['真實证型'] in r['Native_LLM'][:1]) / len(results)
            acc_sh = sum(1 for r in results if r['真實证型'] in r['HoloTSH_V6.7'][:1]) / len(results)
            print(f"\n📈 [階段統計] Native Hit@1: {acc_raw*100:.1f}% -> Guardrail Hit@1: {acc_sh*100:.1f}%")

    print("\n🏁 實驗圓滿完成！數據已存盤。")

if __name__ == "__main__":
    main()

Mounted at /content/drive
🚀 HoloTSH V7.1: gemini-3.1-pro-preview David vs. Goliath Final Trial

💎 [Step 1] 加載流形數據庫 (V7.1 繁簡兼容版)...

👉 [處理 1/100] 靶點: 【阳虚】
    gemini-3.1-pro-preview 返回 (簡體歸一化): ['脾肾阳虚', '心脾两虚', '气血两虚', '阳虚水泛', '水气凌心', '脾虚湿困', '气阴两虚', '湿热内蕴', '心肾阳虚', '阴虚火旺', '气虚发热', '痰湿内阻', '肝郁脾虚', '脾胃虚弱', '痰热内扰', '风水相搏', '肝肾阴虚', '脾阳虚', '肾阳虚', '心气虚']
    [詳細評分 Debug]
       脾肾阳虚: cooc_raw=1.00, topo=0.664, final=0.799
       心脾两虚: cooc_raw=0.00, topo=0.000, final=0.000
       气血两虚: cooc_raw=0.00, topo=0.000, final=0.000
       阳虚水泛: cooc_raw=0.00, topo=0.000, final=0.000
       水气凌心: cooc_raw=0.00, topo=0.000, final=0.000
       脾虚湿困: cooc_raw=0.00, topo=0.000, final=0.000
       气阴两虚: cooc_raw=0.00, topo=0.000, final=0.000
       湿热内蕴: cooc_raw=0.00, topo=0.000, final=0.000
       心肾阳虚: cooc_raw=0.00, topo=0.000, final=0.000
       阴虚火旺: cooc_raw=0.00, topo=0.000, final=0.000
       气虚发热: cooc_raw=0.00, topo=0.000, final=0.000
       痰湿内阻: cooc_raw=0.00, topo=0.000, final=0.000
       肝